In [7]:
import pandas as pd

df = pd.read_csv("keywords.csv", encoding="utf-16")

df.head()


,Keyword Stats 2026-02-03 at 15_23_01
0,"January 1, 2025 - December 31, 2025"
1,Keyword\tCurrency\tAvg. monthly searches\tThre...
2,medanta hospital gurgaon\tINR\t33100\t0%\t0%\t...
3,apollo hospital bannerghatta road\tINR\t9900\t...
4,apollo hospital sarita vihar\tINR\t9900\t0%\t-...


In [11]:
df.columns

Index(['Keyword Stats 2026-02-03 at 15_23_01'], dtype='object')

In [ ]:
columns_needed = [
    "Keyword",
    "Avg. monthly searches",
    "Competition",
    "Competition (indexed value)",
    "Top of page bid (low range)",
    "Top of page bid (high range)",
]

# Normalize column names (strip whitespace, BOM, and NBSP if present)
df.columns = pd.Index([str(c).strip().replace('﻿', '').replace('a0', ' ') for c in df.columns])

# Attempt to map requested columns to actual columns using fuzzy matching
import difflib
col_map = {}
for want in columns_needed:
    if want in df.columns:
        col_map[want] = want
    else:
        match = difflib.get_close_matches(want, df.columns, n=1, cutoff=0.7)
        col_map[want] = match[0] if match else None

print('Column mapping:')
for k,v in col_map.items():
    print(f'  {k} -> {v}')

missing = [w for w,m in col_map.items() if m is None]
if missing:
    print("Missing columns (no close match found):", missing)

# Select the columns we found (preserving the requested order)
found_cols = [col_map[w] for w in columns_needed if col_map[w]]
if not found_cols:
    raise KeyError('None of the requested columns were found in the dataframe')
df = df[found_cols]
# Rename actual columns back to the requested names for downstream code
rename_map = {col_map[w]: w for w in columns_needed if col_map[w]}
df = df.rename(columns=rename_map)
df.head()


Missing columns: ['Keyword', 'Avg. monthly searches', 'Competition', 'Competition (indexed value)', 'Top of page bid (low range)', 'Top of page bid (high range)']


In [18]:
numeric_cols = [
    "Avg. monthly searches",
    "Competition (indexed value)",
    "Top of page bid (low range)",
    "Top of page bid (high range)"
]

# Only convert columns that actually exist after mapping
numeric_cols_existing = [c for c in numeric_cols if c in df.columns]
for col in numeric_cols_existing:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows with NA only in the numeric columns we converted (safer)
if numeric_cols_existing:
    df = df.dropna(subset=numeric_cols_existing)

df.head()


,Keyword Stats 2026-02-03 at 15_23_01
0,"January 1, 2025 - December 31, 2025"
1,Keyword\tCurrency\tAvg. monthly searches\tThre...
2,medanta hospital gurgaon\tINR\t33100\t0%\t0%\t...
3,apollo hospital bannerghatta road\tINR\t9900\t...
4,apollo hospital sarita vihar\tINR\t9900\t0%\t-...


In [19]:
# Parse single-column TSV dump if expected columns are missing
if "Avg. monthly searches" not in df.columns:
    s = df.iloc[:, 0].astype(str)
    header_idx = next((i for i, v in s.items() if "Keyword" in v and "Avg. monthly searches" in v), None)
    if header_idx is None:
        raise KeyError("Header row with 'Avg. monthly searches' not found in dataframe")
    parts = s.str.split('\t', expand=True)
    header = [str(x).strip().replace('\xa0', ' ') for x in parts.iloc[header_idx].tolist()]
    df = parts.iloc[header_idx + 1 :].copy().reset_index(drop=True)
    df.columns = header

# Normalize and convert numeric filter columns
for c in ["Avg. monthly searches", "Competition (indexed value)", "Top of page bid (low range)"]:
    if c in df.columns:
        df[c] = (
            df[c]
            .astype(str)
            .str.replace("%", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Build filter conditions only from columns that exist
conds = []
if "Avg. monthly searches" in df.columns:
    conds.append(df["Avg. monthly searches"].between(500, 10000))
if "Competition (indexed value)" in df.columns:
    conds.append(df["Competition (indexed value)"] <= 40)
if "Top of page bid (low range)" in df.columns:
    conds.append(df["Top of page bid (low range)"] >= 0.3)

if conds:
    cond = conds[0]
    for c in conds[1:]:
        cond &= c
    best_keywords = df[cond].copy()
else:
    best_keywords = df.copy()

best_keywords.head()


,Keyword,Currency,Avg. monthly searches,Three month change,YoY change,Competition,Competition (indexed value),Top of page bid (low range),Top of page bid (high range),Ad impression share,...,Searches: Mar 2025,Searches: Apr 2025,Searches: May 2025,Searches: Jun 2025,Searches: Jul 2025,Searches: Aug 2025,Searches: Sep 2025,Searches: Oct 2025,Searches: Nov 2025,Searches: Dec 2025
1,apollo hospital bannerghatta road,INR,9900,0%,0%,Low,22.0,119.95,832.25,,...,8100,8100,9900,12100,9900,6600,8100,9900,12100,9900
2,apollo hospital sarita vihar,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,12100,9900,9900,9900,9900,9900,9900,8100,8100,8100
3,apollo hospital new delhi,INR,1600,-19%,-19%,Low,7.0,7.94,45.95,,...,1600,1600,1300,1300,1600,1900,1600,1600,1300,1300
5,apollo hospital chennai doctors list,INR,4400,-34%,-19%,Low,9.0,10.97,27.41,,...,5400,4400,4400,4400,5400,5400,4400,4400,3600,2900
7,fortis gurgaon appointment,INR,1300,0%,0%,Medium,35.0,19.97,71.75,,...,1300,1600,1000,1300,1300,1300,1300,1000,1000,1000


In [20]:
best_keywords = best_keywords.sort_values(
    by=["Avg. monthly searches", "Competition (indexed value)"],
    ascending=[False, True]
)

best_keywords.head(20)


,Keyword,Currency,Avg. monthly searches,Three month change,YoY change,Competition,Competition (indexed value),Top of page bid (low range),Top of page bid (high range),Ad impression share,...,Searches: Mar 2025,Searches: Apr 2025,Searches: May 2025,Searches: Jun 2025,Searches: Jul 2025,Searches: Aug 2025,Searches: Sep 2025,Searches: Oct 2025,Searches: Nov 2025,Searches: Dec 2025
2,apollo hospital sarita vihar,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,12100,9900,9900,9900,9900,9900,9900,8100,8100,8100
14,apollo sarita vihar,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,12100,9900,9900,9900,9900,9900,9900,8100,8100,8100
249,apollo sarita vihar delhi,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,12100,9900,9900,9900,9900,9900,9900,8100,8100,8100
356,apollo hospital sarita vihar new delhi,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,12100,9900,9900,9900,9900,9900,9900,8100,8100,8100
1,apollo hospital bannerghatta road,INR,9900,0%,0%,Low,22.0,119.95,832.25,,...,8100,8100,9900,12100,9900,6600,8100,9900,12100,9900
31,medanta gurugram,INR,5400,-19%,0%,Low,9.0,8.17,63.61,,...,6600,6600,5400,4400,5400,5400,6600,5400,4400,4400
1819,good hospitals in india,INR,4400,0%,-19%,Low,2.0,20.86,72.35,,...,4400,4400,4400,5400,5400,5400,4400,4400,4400,4400
2269,top indian hospitals,INR,4400,0%,-19%,Low,2.0,20.86,72.35,,...,4400,4400,4400,5400,5400,5400,4400,4400,4400,4400
1948,leading cardiologist in india,INR,4400,-18%,-18%,Low,6.0,15.19,49.43,,...,5400,4400,4400,5400,5400,5400,4400,4400,4400,3600
5,apollo hospital chennai doctors list,INR,4400,-34%,-19%,Low,9.0,10.97,27.41,,...,5400,4400,4400,4400,5400,5400,4400,4400,3600,2900


In [21]:
best_keywords = df[
    (df["Avg. monthly searches"] >= 500) &
    (df["Avg. monthly searches"] <= 10000) &
    (df["Competition (indexed value)"] <= 40) &
    (df["Top of page bid (low range)"] >= 0.3)
]


In [22]:
best_keywords = best_keywords.sort_values(
    by=["Avg. monthly searches", "Competition (indexed value)"],
    ascending=[False, True]
)

best_keywords.head(20)


,Keyword,Currency,Avg. monthly searches,Three month change,YoY change,Competition,Competition (indexed value),Top of page bid (low range),Top of page bid (high range),Ad impression share,...,Searches: Mar 2025,Searches: Apr 2025,Searches: May 2025,Searches: Jun 2025,Searches: Jul 2025,Searches: Aug 2025,Searches: Sep 2025,Searches: Oct 2025,Searches: Nov 2025,Searches: Dec 2025
2,apollo hospital sarita vihar,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,12100,9900,9900,9900,9900,9900,9900,8100,8100,8100
14,apollo sarita vihar,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,12100,9900,9900,9900,9900,9900,9900,8100,8100,8100
249,apollo sarita vihar delhi,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,12100,9900,9900,9900,9900,9900,9900,8100,8100,8100
356,apollo hospital sarita vihar new delhi,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,12100,9900,9900,9900,9900,9900,9900,8100,8100,8100
1,apollo hospital bannerghatta road,INR,9900,0%,0%,Low,22.0,119.95,832.25,,...,8100,8100,9900,12100,9900,6600,8100,9900,12100,9900
31,medanta gurugram,INR,5400,-19%,0%,Low,9.0,8.17,63.61,,...,6600,6600,5400,4400,5400,5400,6600,5400,4400,4400
1819,good hospitals in india,INR,4400,0%,-19%,Low,2.0,20.86,72.35,,...,4400,4400,4400,5400,5400,5400,4400,4400,4400,4400
2269,top indian hospitals,INR,4400,0%,-19%,Low,2.0,20.86,72.35,,...,4400,4400,4400,5400,5400,5400,4400,4400,4400,4400
1948,leading cardiologist in india,INR,4400,-18%,-18%,Low,6.0,15.19,49.43,,...,5400,4400,4400,5400,5400,5400,4400,4400,4400,3600
5,apollo hospital chennai doctors list,INR,4400,-34%,-19%,Low,9.0,10.97,27.41,,...,5400,4400,4400,4400,5400,5400,4400,4400,3600,2900


In [23]:
best_keywords["Word_Count"] = best_keywords["Keyword"].apply(lambda x: len(str(x).split()))

long_tail = best_keywords[best_keywords["Word_Count"] >= 3]
long_tail.head(20)


,Keyword,Currency,Avg. monthly searches,Three month change,YoY change,Competition,Competition (indexed value),Top of page bid (low range),Top of page bid (high range),Ad impression share,...,Searches: Apr 2025,Searches: May 2025,Searches: Jun 2025,Searches: Jul 2025,Searches: Aug 2025,Searches: Sep 2025,Searches: Oct 2025,Searches: Nov 2025,Searches: Dec 2025,Word_Count
2,apollo hospital sarita vihar,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,9900,9900,9900,9900,9900,9900,8100,8100,8100,4
14,apollo sarita vihar,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,9900,9900,9900,9900,9900,9900,8100,8100,8100,3
249,apollo sarita vihar delhi,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,9900,9900,9900,9900,9900,9900,8100,8100,8100,4
356,apollo hospital sarita vihar new delhi,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,9900,9900,9900,9900,9900,9900,8100,8100,8100,6
1,apollo hospital bannerghatta road,INR,9900,0%,0%,Low,22.0,119.95,832.25,,...,8100,9900,12100,9900,6600,8100,9900,12100,9900,4
1819,good hospitals in india,INR,4400,0%,-19%,Low,2.0,20.86,72.35,,...,4400,4400,5400,5400,5400,4400,4400,4400,4400,4
2269,top indian hospitals,INR,4400,0%,-19%,Low,2.0,20.86,72.35,,...,4400,4400,5400,5400,5400,4400,4400,4400,4400,3
1948,leading cardiologist in india,INR,4400,-18%,-18%,Low,6.0,15.19,49.43,,...,4400,4400,5400,5400,5400,4400,4400,4400,3600,4
5,apollo hospital chennai doctors list,INR,4400,-34%,-19%,Low,9.0,10.97,27.41,,...,4400,4400,4400,5400,5400,4400,4400,3600,2900,5
174,chennai apollo doctor list,INR,4400,-34%,-19%,Low,9.0,10.97,27.41,,...,4400,4400,4400,5400,5400,4400,4400,3600,2900,4


In [24]:
long_tail["Opportunity_Score"] = (
    long_tail["Avg. monthly searches"] /
    (long_tail["Competition (indexed value)"] + 1)
)

long_tail.sort_values(by="Opportunity_Score", ascending=False).head(20)


C:\Users\Main\AppData\Local\Temp\ipykernel_19736\4113777779.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  long_tail["Opportunity_Score"] = (


,Keyword,Currency,Avg. monthly searches,Three month change,YoY change,Competition,Competition (indexed value),Top of page bid (low range),Top of page bid (high range),Ad impression share,...,Searches: May 2025,Searches: Jun 2025,Searches: Jul 2025,Searches: Aug 2025,Searches: Sep 2025,Searches: Oct 2025,Searches: Nov 2025,Searches: Dec 2025,Word_Count,Opportunity_Score
516,big hospitals in india,INR,3600,-19%,-19%,Low,0.0,41.25,205.18,,...,3600,5400,4400,4400,3600,3600,3600,2900,4,3600.000000
1873,india's big hospital,INR,2900,-17%,0%,Low,0.0,33.40,255.25,,...,2900,3600,2900,4400,3600,2900,2900,2400,3,2900.000000
1819,good hospitals in india,INR,4400,0%,-19%,Low,2.0,20.86,72.35,,...,4400,5400,5400,5400,4400,4400,4400,4400,4,1466.666667
2269,top indian hospitals,INR,4400,0%,-19%,Low,2.0,20.86,72.35,,...,4400,5400,5400,5400,4400,4400,4400,4400,3,1466.666667
2,apollo hospital sarita vihar,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,9900,9900,9900,9900,9900,8100,8100,8100,4,1237.500000
249,apollo sarita vihar delhi,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,9900,9900,9900,9900,9900,8100,8100,8100,4,1237.500000
356,apollo hospital sarita vihar new delhi,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,9900,9900,9900,9900,9900,8100,8100,8100,6,1237.500000
14,apollo sarita vihar,INR,9900,0%,-18%,Low,7.0,4.14,56.04,,...,9900,9900,9900,9900,9900,8100,8100,8100,3,1237.500000
1640,dr swaroop gopal bangalore,INR,1900,0%,19%,Low,1.0,28.26,65.94,,...,1600,1900,2400,1900,1900,1900,1900,1900,4,950.000000
623,doctor swaroop gopal,INR,1900,0%,19%,Low,1.0,28.26,65.94,,...,1600,1900,2400,1900,1900,1900,1900,1900,3,950.000000


In [25]:
long_tail.to_csv("best_keywords_final.csv", index=False)
